### Régression du score de toxicité 
On infère le modèle [DistilBERT-toxicity](https://huggingface.co/citizenlab/distilbert-base-multilingual-cased-toxicity) sur l'ensemble des messages politiques (`flat_political_interactions`) et on produit un dataset avec score de toxicité (`flat_toxic_interactions`).

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import pipeline

### Importation du dataset

In [ ]:
df = pd.read_csv("../clean_data/flat_political_interactions.csv")

### Inférence

In [ ]:
classifier = pipeline(
    "text-classification",
    model="citizenlab/distilbert-base-multilingual-cased-toxicity",
    batch_size=64,
    device=-1
)

comments = df["text"].fillna("").tolist()

results = classifier(
    comments,
    truncation=True,
    max_length=256
)

df["toxicity_score"] = [x["score"] for x in results]
df["toxicity_label"] = [x["label"] for x in results]

Sauvegarde du score de toxicité

In [ ]:
df["toxicity_level"] = np.where(
    df["toxicity_label"] == "toxic",
    df["toxicity_score"],
    1 - df["toxicity_score"]
)
df = df.drop(columns=["toxicity_label"])

Exportation

In [ ]:
df.to_csv("../clean_data/flat_toxic_interactions.csv", index=False)